# Data-type scaling

Same `N = 50_000` vertices/nodes across all six geometry types.
Synthetic inputs are tiny per-type generators (random positions / a
spanning tree / a triangulated grid).

Numbers here are *not* a fair cross-type comparison of any
underlying algorithm — different types do genuinely different work.
The point is to see relative per-type cost on the write/read path.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.types.lines import write_lines, read_lines
from zarr_vectors.types.polylines import write_polylines, read_polylines
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.types.meshes import write_mesh, read_mesh

N = 50_000
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0
rng = np.random.default_rng(SEED)

## 2 · Synthetic generators (one per type)

In [ ]:
def _points_input():
    return rng.uniform(0, 1000, (N, 3)).astype(np.float32)


def _lines_input():
    # N lines, each two random endpoints
    return rng.uniform(0, 1000, (N, 2, 3)).astype(np.float32)


def _polylines_input():
    # ~N total vertices spread across short random walks
    counts = rng.integers(8, 16, size=N // 12)
    out = []
    for c in counts:
        steps = rng.normal(0, 5, (c, 3))
        start = rng.uniform(0, 1000, 3)
        out.append((start + steps.cumsum(axis=0)).astype(np.float32))
    return out


def _graph_input(is_tree=False):
    positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)
    if is_tree:
        # spanning-tree edges: each node i (i>=1) connects to a random ancestor
        parents = rng.integers(0, np.arange(1, N))
        edges = np.stack([np.arange(1, N), parents], axis=1).astype(np.int64)
    else:
        # ~3N/2 random undirected edges
        src = rng.integers(0, N, size=3 * N // 2)
        dst = rng.integers(0, N, size=3 * N // 2)
        mask = src != dst
        edges = np.stack([src[mask], dst[mask]], axis=1).astype(np.int64)
    return positions, edges


def _mesh_input():
    # Triangulated grid with ~N vertices
    side = int(np.sqrt(N))
    xs, ys = np.meshgrid(np.linspace(0, 1000, side), np.linspace(0, 1000, side))
    zs = rng.uniform(0, 100, (side, side))
    verts = np.stack([xs, ys, zs], axis=-1).reshape(-1, 3).astype(np.float32)
    # Two triangles per grid cell.
    i = np.arange(side - 1)
    j = np.arange(side - 1)
    ii, jj = np.meshgrid(i, j, indexing='ij')
    a = (ii * side + jj).ravel()
    b = a + 1
    c = a + side
    d = c + 1
    faces = np.concatenate([
        np.stack([a, b, c], axis=1),
        np.stack([b, d, c], axis=1),
    ]).astype(np.int64)
    return verts, faces

## 3 · Run the sweep

In [ ]:
def bench_points():
    pts = _points_input()
    store = _new_store('points')
    tw, _ = _time(write_points, store, pts, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _ = _time(read_points, store)
    return tw, tr, _store_bytes(store), store

def bench_lines():
    eps = _lines_input()
    store = _new_store('lines')
    tw, _ = _time(write_lines, store, eps, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _ = _time(read_lines, store)
    return tw, tr, _store_bytes(store), store

def bench_polylines():
    plys = _polylines_input()
    store = _new_store('polylines')
    tw, _ = _time(write_polylines, store, plys, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _ = _time(read_polylines, store)
    return tw, tr, _store_bytes(store), store

def bench_graph(is_tree):
    pos, edges = _graph_input(is_tree=is_tree)
    store = _new_store('skeleton' if is_tree else 'graph')
    tw, _ = _time(
        write_graph, store, pos, edges,
        chunk_shape=CHUNK, bin_shape=BIN, is_tree=is_tree,
    )
    tr, _ = _time(read_graph, store)
    return tw, tr, _store_bytes(store), store

def bench_mesh():
    verts, faces = _mesh_input()
    store = _new_store('mesh')
    tw, _ = _time(write_mesh, store, verts, faces, chunk_shape=CHUNK, bin_shape=BIN)
    tr, _ = _time(read_mesh, store)
    return tw, tr, _store_bytes(store), store


fns = [
    ('points',    bench_points),
    ('lines',     bench_lines),
    ('polylines', bench_polylines),
    ('graph',     lambda: bench_graph(is_tree=False)),
    ('skeleton',  lambda: bench_graph(is_tree=True)),
    ('mesh',      bench_mesh),
]

rows = []
for name, fn in fns:
    tw, tr, nbytes, store = fn()
    rows.append({
        'type':    name,
        'write_s': round(tw, 3),
        'read_s':  round(tr, 3),
        'size_MB': round(nbytes / 1e6, 2),
    })
    shutil.rmtree(Path(store).parent, ignore_errors=True)

df = pd.DataFrame(rows)

## 4 · Results

In [ ]:
df

## 5 · Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(df))
ax.bar(x - 0.18, df['write_s'], width=0.36, label='write (s)')
ax.bar(x + 0.18, df['read_s'],  width=0.36, label='read (s)')
ax.set_xticks(x)
ax.set_xticklabels(df['type'])
ax.set_ylabel('seconds')
ax.set_title(f'Write / read time per type (N = {N:,})')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()